In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']
xu, yu = s['xu'], s['yu']

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()



In [5]:
def calc_fluxes_(file, dlat=5):
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    did = int(file.split('_')[-1].split('.')[0])

    view = View.from_header(header)
    view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

    transform = view.to_carrington(correct_mu=True) + ToSpherical()
    grid, mu = transform(crop_grid(xu, yu, header))

    Br = data / mu
    lat_map = grid[0] // dlat * dlat

    lat = np.arange(-90,90,dlat)
    weight = np.zeros_like(lat).astype(np.float32)
    flux_density = np.zeros_like(lat).astype(np.float32)

    for i, lat_ in enumerate(lat):
        t = (lat_map == lat_)
        nt = np.sum(t)

        if nt > 0:
            Br_ = Br[t]
            W_ = mu[t] ** 4 / mu[t]

            weight[i] = np.nansum(W_)
            flux_density[i] = np.nansum(W_ * Br_) / weight[i]

    return lat, weight, flux_density


def calc_fluxes(files, **kwargs):
    x, mean, w_sum, w_sum2, S = 0., 0., 0. ,0., 0.

    for file in files:
        x, w, y = calc_fluxes_(file, **kwargs)

        w_sum += w
        w_sum2 += w ** 2
        mean_old = mean + 0.
        mean += np.nan_to_num((w / w_sum) * (y - mean))
        S += np.nan_to_num(w * (y - mean_old) * (y - mean))

    S *= w_sum2 / w_sum ** 3
    return x, mean, S


In [6]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))
t = np.where(car_rot == 2302)[0]

x, mean, S = calc_fluxes(files[t])

/tmp/ipykernel_28166/3833533144.py:44: RuntimeWarning: invalid value encountered in divide
  mean += np.nan_to_num((w / w_sum) * (y - mean))


In [7]:
plt.figure(figsize=(10,8))
plt.plot(x, mean)
plt.plot(x, np.sqrt(S))

plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()


In [8]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))

flux_density, variance = [], []

for i in np.unique(car_rot):
    print(i)

    t = np.where(car_rot == i)[0]
    x, mean, S = calc_fluxes(files[t], dlat=5)

    flux_density += [mean]
    variance += [S]

lat = x
flux_density = np.array(flux_density)
variance = np.array(variance)

2279
2280
2281
2282


/tmp/ipykernel_28166/3833533144.py:44: RuntimeWarning: invalid value encountered in divide
  mean += np.nan_to_num((w / w_sum) * (y - mean))


2283


/tmp/ipykernel_28166/3833533144.py:47: RuntimeWarning: invalid value encountered in divide
  S *= w_sum2 / w_sum ** 3


2284
2285
2286
2287
2288
2289
2290
2291
2292
2293
2294
2295
2296
2297
2298
2299
2300
2301
2302
2303
2304
2305
2306


In [9]:
np.savez('fluxes.npz', lat=lat, flux_density=flux_density, variance=variance, car_rot=np.unique(car_rot))

In [41]:
s = np.load('fluxes.npz')
lat = s['lat']
flux_density = s['flux_density']
variance = s['variance']
q = np.nan_to_num(1 - np.exp(-flux_density ** 2 / 2 / variance))
car_rot = s['car_rot']

plt.figure(figsize=(12,8))
#plt.imshow(flux_density.T, vmin=-4, vmax=4, origin='lower', cmap='bwr', aspect='auto')#, interpolation='bicubic')
plt.imshow(q.T, vmin=0, vmax=1, origin='lower', cmap='jet', aspect='auto')#, interpolation='bicubic')

plt.xlabel('Carrington rotation number')
plt.ylabel('Latitude, degrees')

plt.yticks((lat[::3] + 90) / 5 - 0.5, lat[::3])
plt.xticks(np.arange(len(car_rot)) - 0.5, car_rot, rotation='vertical', size='small')

plt.colorbar(label=r'Magnetic flux density, $Mx / cm^2$', pad=0.02, aspect=50)
#plt.grid(True)
plt.tight_layout()

In [46]:
plt.figure(figsize=(10,8))
plt.plot(x + 2.5, flux_density[23])
plt.plot(x + 2.5, variance[23])

plt.xlim(-90,90)
plt.ylim(-5,5)
plt.grid(True)
plt.tight_layout()